<a href="https://colab.research.google.com/github/zamanuddinkhan/Python-AI-LLM/blob/main/LangChain_Basics_Memory_Complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangChain Memory Types — Real-Life Examples
### Powered by ChatGroq (llama-3.1-8b-instant)

| # | Memory Type | One-line idea |
|---|---|---|
| 1 | `ConversationBufferMemory` | Remembers everything |
| 2 | `ConversationBufferWindowMemory` | Remembers only the last K turns |
| 3 | `ConversationSummaryMemory` | Keeps a running summary |
| 4 | `ConversationSummaryBufferMemory` | Summary + recent messages |
| 5 | `ConversationTokenBufferMemory` | Drops old messages by token count |
| 6 | `ConversationEntityMemory` | Tracks named people and places |

## Setup

In [ ]:
!pip install langchain==0.2.17 langchain-groq langchain-community -q

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] =userdata.get('GROQ_API_KEY')

In [ ]:
from langchain_groq import ChatGroq
from langchain.chains import ConversationChain

llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.7)
print("LLM ready")

---
## 1. ConversationBufferMemory

Stores every message verbatim — nothing is ever dropped.

**Scenario:** Shopping assistant that must recall everything said in the session.

In [ ]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory()
chat   = ConversationChain(llm=llm, memory=memory, verbose=False)

chat.predict(input="My name is Sara. I need a gift for my husband who loves coffee.")
chat.predict(input="Budget is around 500 rupees.")
reply = chat.predict(input="What gift would you suggest based on what I told you?")
print(reply)

In [ ]:
print(memory.buffer)

In [ ]:
reply = chat.predict(input="What is my budget ?")
print(reply)

In [ ]:
reply = chat.predict(input="what is the Capital of France ?")
print(reply)

In [ ]:
print(memory.buffer)

---
## 2. ConversationBufferWindowMemory

Keeps only the last K turns. Older messages are dropped.

**Scenario:** Support bot where only the most recent exchange matters.

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(k=2)
chat   = ConversationChain(llm=llm, memory=memory, verbose=False)

chat.predict(input="My order number is 12345.")          # will be forgotten
chat.predict(input="The package arrived damaged.")
chat.predict(input="It was not the product that I ordered")
reply = chat.predict(input="What was my order number?")  # bot will not know
print(reply)

In [ ]:
print(memory.buffer)

---
## 3. ConversationSummaryMemory

After each turn the LLM rewrites the conversation as a short summary.

**Scenario:** Travel planner where a running brief is more useful than raw logs.

In [ ]:
from langchain.memory import ConversationSummaryMemory

memory = ConversationSummaryMemory(llm=llm)
chat   = ConversationChain(llm=llm, memory=memory, verbose=False)

chat.predict(input="I want to visit India for 10 days in April.")
chat.predict(input="I prefer vegetarian food and love temples.")
reply = chat.predict(input="Suggest a rough itinerary based on what I shared.")
print(reply)

In [ ]:
print(memory.buffer)

---
## 4. ConversationSummaryBufferMemory

Recent messages stay raw; older ones get summarised once the token limit is reached.

**Scenario:** Medical assistant that needs exact recent symptoms but can summarise earlier background.

In [ ]:
from langchain.memory import ConversationSummaryBufferMemory

memory = ConversationSummaryBufferMemory(llm=llm, max_token_limit=200)
chat   = ConversationChain(llm=llm, memory=memory, verbose=False)

chat.predict(input="I have had headaches on the right side for a week.")
chat.predict(input="Pain level is 6 out of 10, worse in the morning.")
reply = chat.predict(input="Should I see a neurologist?")
print(reply)

In [ ]:
print("Summary:", memory.moving_summary_buffer or "(none yet)")
print("\nRecent raw messages:")
for m in memory.chat_memory.messages[-2:]:
    print(f"  [{m.type}]: {m.content[:80]}")

---
## 5. ConversationTokenBufferMemory

Drops the oldest messages when the token budget is exceeded. No summarisation.

**Scenario:** Recipe assistant with a strict token limit.

In [ ]:
from langchain.memory import ConversationTokenBufferMemory

memory = ConversationTokenBufferMemory(llm=llm, max_token_limit=300)
chat   = ConversationChain(llm=llm, memory=memory, verbose=False)

reply=chat.predict(input="Give me a quick pasta recipe.")           # may be pruned
print(reply)
chat.predict(input="Now I want to make a chocolate cake.")
reply = chat.predict(input="What temperature should I bake it at?")
print(reply)

In [ ]:
print(memory.buffer)

In [ ]:
msgs = memory.chat_memory.messages
print(f"Messages retained: {len(msgs)}")
for m in msgs:
    print(f"  [{m.type}]: {m.content[:80]}")

---
## 6. ConversationEntityMemory

Extracts named entities and builds a structured fact store about each one.

**Scenario:** Networking assistant that remembers details about the people you mention.

In [ ]:
from langchain.memory import ConversationEntityMemory
from langchain.prompts import PromptTemplate

memory = ConversationEntityMemory(llm=llm)

prompt = PromptTemplate(
    input_variables=["history", "entities", "input"],
    template=(
        "You are a helpful assistant.\n\n"
        "Known entities:\n{entities}\n\n"
        "Chat so far:\n{history}\n\n"
        "Human: {input}\nAI:"
    )
)

chat = ConversationChain(llm=llm, memory=memory, prompt=prompt, verbose=False)

chat.predict(input="Rahul is a product manager at Zomato in Bangalore.")
chat.predict(input="Priya just joined Google as an SRE.")
reply = chat.predict(input="What do you know about Rahul and Priya?")
print(reply)

In [ ]:
for name, info in memory.entity_store.store.items():
    print(f"{name}: {info}")

In [ ]:
print(memory.buffer)

---
## Quick Cheat Sheet

| Memory Type | Stores | Forgets | Best For |
|---|---|---|---|
| `BufferMemory` | Everything verbatim | Nothing | Short sessions |
| `BufferWindowMemory` | Last K turns | Old turns | Support bots |
| `SummaryMemory` | LLM summary | Raw messages | Long topic chats |
| `SummaryBufferMemory` | Summary + recent raw | Older raw messages | Medical / legal bots |
| `TokenBufferMemory` | Recent by token count | Oldest messages | Budget-constrained apps |
| `EntityMemory` | Named entity facts | Nothing about entities | CRM / personal assistants |

> For most production chatbots, `SummaryBufferMemory` or `TokenBufferMemory` gives the best balance of context quality and cost.